In [78]:
import pandas as pd
import numpy as np
import ast
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

from scipy.sparse import csr_matrix, hstack

In [79]:
df = pd.read_csv(r"D:\Phishing Email\Nazario_5.csv")

In [80]:
df

,sender,receiver,date,subject,body,label,urls
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr\nPW: bnaweb22\n\n\n ...,0,"['http://web.bna.com', 'http://pubs.bna.com/ip..."
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,[]
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,0,['http://eastpower.dev.corp.enron.com/summary/...
3,"""Thorne, Judy"" <Judy.Thorne@ENRON.com>","""Grass, John"" <John.Grass@ENRON.com>, ""Nemec, ...","Fri, 29 Jun 2001 10:35:17 -0500",FW: ENA Upstream Company information,"John/Gerald,\n\nWe are currently trading under...",0,[]
4,"""Williams, Jason R (Credit)"" <Jason.R.Williams...","""Nemec, Gerald"" <Gerald.Nemec@ENRON.com>, ""Dic...","Fri, 29 Jun 2001 10:40:02 -0500",New Master Physical,Gerald and Stacy -\n\nAttached is a worksheet ...,0,[]
...,...,...,...,...,...,...,...
3060,info advantageapartments.com <info@advantageap...,"""jose@monkey.org"" <jose@monkey.org>","Sun, 11 Dec 2022 12:04:50 +0000",Receipt for Your Payment to FTX.,PayPal You sent a payment of $699.99 USD to FT...,1,0
3061,monkey.org <HELPDESK@monkey.org>,jose@monkey.org,15 Dec 2022 18:33:00 -0500,Rectify Your Password With monkey.org,"monkey.org Hi jose,Pa⁠s⁠sword for⁠ jose@monke...",1,1
3062,Help Center\n\t<info.help-center.co.za_info.he...,jose@monkey.org,"Tue, 20 Dec 2022 23:00:27 +0000 (UTC)",Netflix : We're having some trouble with your ...,"HELLO, Please note that, your monthly paymen...",1,1
3063,MetaMask <info@sofamekar.com>,jose@monkey.org,27 Dec 2022 10:56:49 +0800,Your MetaMask wallet will be suspended,Verify your MetaMask Wallet Our system has sho...,1,1


In [81]:
df.shape

(3065, 7)

In [82]:
print(df["label"].value_counts())

label
1    1565
0    1500
Name: count, dtype: int64


In [83]:
df["text"] = df["subject"].fillna("") + " " + df["body"].fillna("")

In [84]:
df["sender_domain"] = df["sender"].str.extract(r'@([A-Za-z0-9.-]+)')
df["sender_domain"] = df["sender_domain"].str.lower()

In [85]:
def count_urls(x):
    try:
        if isinstance(x, list):
            return len(x)
        if isinstance(x, str):
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return len(parsed)
    except:
        pass
    return 0

df["url_count"] = df["urls"].apply(count_urls)

In [86]:
import ast
import re

def extract_url_features(x):

    
    urls = []

    try:
        
        if isinstance(x, list):
            urls = x

        
        elif isinstance(x, str):
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                urls = parsed

       
        else:
            urls = []

    except:
        urls = []

    
    if not urls:
        return [0, 0, 0, 0]

    
    url = str(urls[0])

    length = len(url)
    dots = url.count(".")
    has_ip = int(bool(re.search(r"\d+\.\d+\.\d+\.\d+", url)))
    has_at = int("@" in url)

    return [length, dots, has_ip, has_at]

In [87]:
url_features = df["urls"].apply(extract_url_features)

url_df = pd.DataFrame(
    url_features.tolist(),
    columns=["url_len","url_dots","url_ip","url_at"]
)

df = pd.concat([df, url_df], axis=1)

In [88]:
url_features = df["urls"].apply(extract_url_features)

In [89]:
def capital_ratio(text):
    if len(text)==0:
        return 0
    return sum(1 for c in text if c.isupper()) / len(text)

df["capital_ratio"] = df["text"].apply(capital_ratio)

In [90]:
df["exclamation_count"] = df["text"].str.count("!")

In [91]:
df["text_length"] = df["text"].str.len()

In [92]:
feature_df = df[[
    "text",
    "url_count",
    "url_len",
    "url_dots",
    "url_ip",
    "url_at",
    "capital_ratio",
    "exclamation_count",
    "text_length",
    "label"
]].copy()

In [93]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    feature_df["text"],
    feature_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=feature_df["label"]
)

In [94]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

In [95]:
extra_cols = [
    "url_count","url_len","url_dots",
    "url_ip","url_at",
    "capital_ratio","exclamation_count","text_length"
]

X_train_extra = csr_matrix(
    feature_df.loc[X_train_text.index, extra_cols].values
)

X_test_extra = csr_matrix(
    feature_df.loc[X_test_text.index, extra_cols].values
)

In [96]:
X_train = hstack([X_train_tfidf, X_train_extra])
X_test = hstack([X_test_tfidf, X_test_extra])

In [97]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [98]:
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.9918433931484503
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       300
           1       0.99      0.99      0.99       313

    accuracy                           0.99       613
   macro avg       0.99      0.99      0.99       613
weighted avg       0.99      0.99      0.99       613



In [99]:
from scipy.sparse import hstack, csr_matrix
import re

def analyze_email(email_text, url=""):

    text_vec = tfidf.transform([email_text])

    
    if url:
        url_len = len(url)
        url_dots = url.count(".")
        url_ip = int(bool(re.search(r"\d+\.\d+\.\d+\.\d+", url)))
        url_at = int("@" in url)
        url_count = 1
    else:
        url_len = 0
        url_dots = 0
        url_ip = 0
        url_at = 0
        url_count = 0

    
    text_len = len(email_text)
    caps = capital_ratio(email_text)
    exclam = email_text.count("!")

    extra = csr_matrix([[
        url_count,
        url_len,
        url_dots,
        url_ip,
        url_at,
        caps,
        exclam,
        text_len
    ]])

    X_input = hstack([text_vec, extra])

    pred = model.predict(X_input)[0]
    prob = model.predict_proba(X_input)[0][1]

    print("Prediction:", "PHISHING ⚠️" if pred==1 else "SAFE ✅")
    print("Risk Score:", round(prob*100,2), "%")

In [100]:
analyze_email(
"URGENT! Your bank account has been suspended. Click to verify now.",
url="http://secure-bank-login-verification.com/login"
)

Prediction: PHISHING ⚠️
Risk Score: 54.0 %


In [101]:
analyze_email(
"Dear customer, we detected unusual activity in your bank account. Please login immediately to confirm your identity and avoid temporary suspension.",
url="http://secure-update-account-login.com/verify"
)

Prediction: PHISHING ⚠️
Risk Score: 59.5 %
